# Libraries 

In [1]:
from screendl.screendl import model as screendl
import keras
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import scipy.stats as stats

2026-01-29 17:52:46.366176: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-29 17:52:46.582365: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-01-29 17:52:46.582387: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-01-29 17:52:47.362853: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2026-

In [ ]:
def precision_at_q(y_true, y_hat, q=0.25):
    y_true = np.asarray(y_true)
    y_hat = np.asarray(y_hat)

    thr = np.quantile(y_true, q)
    true_pos = np.where(y_true <= thr)[0]
    k = len(true_pos)

    pred_topk = np.argsort(y_hat)[:k]

    return len(set(true_pos) & set(pred_topk)) / k

def ndcg_at_q(y_true, y_hat, q=0.25):
    "Normalized discounted cumulative gain"
    y_true = np.asarray(y_true)
    y_hat = np.asarray(y_hat)

    thr = np.quantile(y_true, q)
    true_pos = np.where(y_true <= thr)[0]
    k = len(true_pos)

    # relevance: higher is better
    rel = -y_true
    
    # predicted ranking
    order = np.argsort(y_hat)
    rel_pred = rel[order][:k]
    
    discounts = 1 / np.log2(np.arange(2, k + 2)) # rank weight
    dcg = np.sum((2 ** rel_pred - 1) * discounts)

    # ideal ranking
    ideal_order = np.argsort(y_true)
    rel_ideal = rel[ideal_order][:k]
    idcg = np.sum((2 ** rel_ideal - 1) * discounts)

    return dcg / idcg if idcg > 0 else 0.0

# Cross Validation

In [2]:
epochs= 100
batch_size= 256
learning_rate= 0.0001
weight_decay= 1e-4
earl_stopping=True


In [3]:
expression_data = pd.read_csv("../../output/regression/msigdb_GEX_data_filtered_logCPM.csv", header=0, index_col=0)

expression_data = pd.read_csv("../../output/regression/GEX_data_filtered_logCPM.csv", header=0, index_col=0)

In [4]:
vector_smiles = pd.read_csv("../../output/regression/vector_smiles_512.csv", header=0, index_col=0)

## NBS Cells

In [6]:
root = "../../output/regression/cross_validation/"
cancer_type = "pancancer" # pancancer solid_tumors

for n in range(10):
    
    fold = f"{root}/{cancer_type}/NBS_cells/fold_{n}"
    
    # Train set
    ## Indices
    train_set = pd.read_csv(f"{fold}/train_set.csv", index_col=0)
       
    ## GEX
    train_exp = expression_data.loc[train_set.CELL_LINE_NAME]
    tx_mean = train_exp.mean(0)
    tx_std = train_exp.std(0)
    X = (train_exp-tx_mean)/tx_std
    X = X.to_numpy(dtype="float16")
    
    ## SMILES
    train_sm = vector_smiles.loc[train_set.DRUG_NAME].to_numpy(dtype="float16")
    
    ## Y TRUE
    ic50_mean = train_set.groupby("DRUG_NAME")["LN_IC50"].mean().to_dict()
    ic50_std = train_set.groupby("DRUG_NAME")["LN_IC50"].std().to_dict()
    
    y = (train_set["LN_IC50"]-train_set["DRUG_NAME"].map(ic50_mean))/train_set["DRUG_NAME"].map(ic50_std).to_numpy(dtype="float16")
    
    train_ds = tf.data.Dataset.from_tensor_slices(((X, train_sm), y)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    # Val set
    val_set = pd.read_csv(f"{fold}/val_set.csv", index_col=0)
    ## GEX
    val_x = expression_data.loc[val_set.CELL_LINE_NAME]
    val_x = (val_x-tx_mean)/tx_std
    val_x = val_x.to_numpy(dtype="float16")
    
    ## SMILES
    val_sm = vector_smiles.loc[val_set.DRUG_NAME].to_numpy(dtype="float16")
    
    ## Y TRUE
    val_y = (val_set["LN_IC50"]-val_set["DRUG_NAME"].map(ic50_mean))/val_set["DRUG_NAME"].map(ic50_std).to_numpy(dtype="float16")
    
    val_ds = tf.data.Dataset.from_tensor_slices(((val_x, val_sm), val_y)).batch(batch_size)

    ######## model
    model = screendl.create_model(
        exp_dim=4377, #946 | 4377
        mol_dim=512,
        mut_dim=None,
        cnv_dim=None,
        ont_dim=None,
        exp_hidden_dims=[512, 256, 128, 64],
        mut_hidden_dims=None,
        cnv_hidden_dims=None,
        ont_hidden_dims=None,
        mol_hidden_dims=[256, 128, 64],
        shared_hidden_dims=[128, 64],
        use_mr=False,
        use_noise=True,
        use_batch_norm=False,
        use_dropout=False,
        use_l2=False,
        noise_stddev=0.3,
        l2_factor=1e-4,
        dropout_rate=0.0,
        activation='leaky_relu'
    )

    opt = keras.optimizers.Adam(
        learning_rate=learning_rate, decay=weight_decay
    )
    model.compile(optimizer=opt, loss="mean_squared_error")

    model.fit(
    train_ds,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=val_ds, 
    workers=22, 
    use_multiprocessing=True
    )

    model.save(f"models_{cancer_type}_NBS_cells/screendl_model_{n}.keras")

    del model, opt

    print(f"Finished fold: {n}")

2025-11-05 10:54:33.579823: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-11-05 10:54:33.582972: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-11-05 10:54:33.583053: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory
2025-11-05 10:54:33.583114: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublasLt.so.11'; dlerror: libcublasLt.so.11: cannot open shared object file: No such file or directory
2025-11-05 10:54:33.583173: W tensorflow/c

Epoch 1/100
850/850 [==============================] - 14s 16ms/step - loss: 0.7652 - val_loss: 0.7371
Epoch 2/100
850/850 [==============================] - 13s 15ms/step - loss: 0.7043 - val_loss: 0.6979
Epoch 3/100
850/850 [==============================] - 13s 16ms/step - loss: 0.6586 - val_loss: 0.6573
Epoch 4/100
850/850 [==============================] - 13s 15ms/step - loss: 0.6216 - val_loss: 0.6301
Epoch 5/100
850/850 [==============================] - 13s 15ms/step - loss: 0.5960 - val_loss: 0.6131
Epoch 6/100
850/850 [==============================] - 13s 15ms/step - loss: 0.5773 - val_loss: 0.6012
Epoch 7/100
850/850 [==============================] - 13s 15ms/step - loss: 0.5638 - val_loss: 0.5929
Epoch 8/100
850/850 [==============================] - 13s 15ms/step - loss: 0.5524 - val_loss: 0.5861
Epoch 9/100
850/850 [==============================] - 12s 15ms/step - loss: 0.5428 - val_loss: 0.5809
Epoch 10/100
850/850 [==============================] - 13s 15ms/step - l

KeyboardInterrupt: 

# NBS Cells Evaluation

In [3]:
expression_data = pd.read_csv("../../output/regression/msigdb_GEX_data_filtered_logCPM.csv", header=0, index_col=0)

In [4]:
vector_smiles = pd.read_csv("../../output/regression/vector_smiles_512.csv", header=0, index_col=0)

In [6]:
epochs= 100
batch_size= 256
learning_rate= 0.0001
weight_decay= 1e-4
earl_stopping=True


## Fixed-drug and fixed-cells

In [7]:
cancer_type = "pancancer"
root = f"../../output/regression/cross_validation/{cancer_type}/NBS_cells"

per_drug_pcc_cv = {}
per_cell_pcc_cv = {}

per_drug_precision_cv = {}
per_cell_precision_cv = {}

per_drug_ndcg_cv = {}
per_cell_ndcg_cv = {}

for n in range(10):
    fold_name = f"fold_{n}"
    fold = f"{root}/fold_{n}"
    
    # Train stats
    train_set = pd.read_csv(f"{fold}/train_set.csv", index_col=0)
    ## GEX
    train_exp = expression_data.loc[train_set.CELL_LINE_NAME]
    tx_mean = train_exp.mean(0)
    tx_std = train_exp.std(0)
    
    ## IC50
    ic50_mean = train_set.groupby("DRUG_NAME")["LN_IC50"].mean().to_dict()
    ic50_std = train_set.groupby("DRUG_NAME")["LN_IC50"].std().to_dict()
    
    # Read files
    test_set = pd.read_csv(f"{fold}/test_set.csv", index_col=0)
    gex = expression_data.loc[test_set.CELL_LINE_NAME]
    gex = (gex-tx_mean)/tx_std
    
    smiles = vector_smiles.loc[test_set.DRUG_NAME]
    
    test_set["Y_TRUE"] = (test_set["LN_IC50"]-test_set["DRUG_NAME"].map(ic50_mean))/test_set["DRUG_NAME"].map(ic50_std)
    
    test_ds = tf.data.Dataset.from_tensor_slices(((gex.to_numpy(), smiles.to_numpy()), )).batch(batch_size)
    
    # Load model
    model = load_model(f"models_{cancer_type}_NBS_cells/screendl_model_{n}.keras")
    
    # Prediction
    y_hat = model.predict(test_ds)
    
    y_hat = y_hat.astype(np.float64)
    
    test_set["Y_HAT"] = y_hat
    
    # Computing metrics
    per_drug_pcc = (test_set.groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    
    per_drug_pcc_cv[fold_name] = [per_drug_pcc["PCC"].median()]
    
    per_cell_pcc = (test_set.groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    
    per_cell_pcc_cv[fold_name] = [per_cell_pcc["PCC"].median()]

    ##############################
    # PRECISION
    ##############################
    per_drug_pcc = (test_set.groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_drug_precision_cv[fold_name] = [per_drug_pcc["precision@q25"].median()]

    per_cell_pcc = (test_set.groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_cell_precision_cv[fold_name] = [per_cell_pcc["precision@q25"].median()]

    ##############################
    # NDCG
    ##############################
    per_drug_pcc = (test_set.groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_drug_ndcg_cv[fold_name] = [per_drug_pcc["ndcg@q25"].median()]

    per_cell_pcc = (test_set.groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_cell_ndcg_cv[fold_name] = [per_cell_pcc["ndcg@q25"].median()]

    

135/135 [==============================] - 2s 7ms/step


/tmp/ipykernel_28463/1460358030.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/1460358030.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/1460358030.py:72: Deprecation

136/136 [==============================] - 2s 7ms/step


/tmp/ipykernel_28463/1460358030.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/1460358030.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/1460358030.py:72: Deprecation

135/135 [==============================] - 2s 7ms/step


...

/tmp/ipykernel_28463/1460358030.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/1460358030.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/1460358030.py:72: Deprecation

135/135 [==============================] - 2s 7ms/step


/tmp/ipykernel_28463/1460358030.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/1460358030.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/1460358030.py:72: Deprecation

134/134 [==============================] - 2s 7ms/step


/tmp/ipykernel_28463/1460358030.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/1460358030.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/1460358030.py:72: Deprecation

In [8]:
per_drug_pcc_cv = pd.DataFrame(per_drug_pcc_cv)
per_drug_pcc_cv.index = ["ScreenDL"]

In [9]:
M = np.median(per_drug_pcc_cv)
sem = stats.sem(per_drug_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.4621357395075028, low=0.44144453420986, high=0.4828269448051456


In [26]:
per_drug_pcc_cv.to_csv(f"predictions_{cancer_type}_NBS_cells/msigbd_fixed-drug_evaluation.csv")

In [10]:
per_cell_pcc_cv = pd.DataFrame(per_cell_pcc_cv)
per_cell_pcc_cv.index = ["ScreenDL"]

In [11]:
M = np.median(per_cell_pcc_cv)
sem = stats.sem(per_cell_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.3391693757779838, low=0.3240299957310476, high=0.35430875582491994


In [25]:
per_cell_pcc_cv.to_csv(f"predictions_{cancer_type}_NBS_cells/msigdb_fixed-cell_evaluation.csv")

In [12]:
per_drug_precision_cv = pd.DataFrame(per_drug_precision_cv)
per_drug_precision_cv.index = ["ScreenDL"]
per_drug_precision_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
ScreenDL,0.521739,0.5,0.5,0.521739,0.5,0.5,0.5,0.545455,0.5,0.454545


In [17]:
per_drug_precision_cv.to_csv(
    "predictions_pancancer_NBS_cells/precision_fixed-drug_CV.csv"
)

In [13]:
per_drug_ndcg_cv = pd.DataFrame(per_drug_ndcg_cv)
per_drug_ndcg_cv.index = ["ScreenDL"]
per_drug_ndcg_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
ScreenDL,0.521833,0.509153,0.488358,0.509057,0.490524,0.500556,0.572676,0.527477,0.489047,0.449285


In [18]:
per_drug_ndcg_cv.to_csv(
    "predictions_pancancer_NBS_cells/ndcg_fixed-drug_CV.csv"
)

In [15]:
per_cell_precision_cv = pd.DataFrame(per_cell_precision_cv)
per_cell_precision_cv.index = ["ScreenDL"]
per_cell_precision_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
ScreenDL,0.384559,0.398413,0.4,0.40625,0.393617,0.393617,0.410526,0.408602,0.417476,0.411111


In [19]:
per_cell_precision_cv.to_csv(
    "predictions_pancancer_NBS_cells/precision_fixed-cell_CV.csv"
)

In [16]:
per_cell_ndcg_cv = pd.DataFrame(per_cell_ndcg_cv)
per_cell_ndcg_cv.index = ["ScreenDL"]
per_cell_ndcg_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
ScreenDL,0.320616,0.363592,0.331264,0.34192,0.332322,0.279182,0.365759,0.298662,0.321259,0.321888


In [20]:
per_cell_ndcg_cv.to_csv(
    "predictions_pancancer_NBS_cells/ndcg_fixed-cell_CV.csv"
)

In [25]:
cv_NBS_cells = pd.DataFrame({
    "mode":["Fixed-drug", "Fixed-cell line"],
    "median": [0.4666041510894773, 0.3283003410671924],
    "low" : [0.4501654223365666, 0.30892653058757824],
    "high" : [0.483042879842388, 0.3476741515468066]
})

In [24]:
import plotly.express as px

In [ ]:
fig = px.scatter(
    cv_NBS_cells,
    x="mode",
    y="median",
    error_y=cv_NBS_cells["high"] - cv_NBS_cells["median"],
    error_y_minus=cv_NBS_cells["median"] - cv_NBS_cells["low"],
    #markers=True,
    title="CV perfromance on NBS cell lines with 95% CI",
    labels={"median": "Median PCC", "mode": ""},
    template="simple_white",
    height=700,
    width=700
)
fig.update_layout(font=dict(size=16))

## Fixed-pathway and fixed-tcga

In [5]:
cancer_type = "pancancer"
root = f"../../output/regression/cross_validation/{cancer_type}/NBS_cells"

per_path_pcc_cv = {}
per_tcga_pcc_cv = {}

for n in range(10):
    fold_name = f"fold_{n}"
    fold = f"{root}/fold_{n}"
    
    # Train stats
    train_set = pd.read_csv(f"{fold}/train_set.csv", index_col=0)
    ## GEX
    train_exp = expression_data.loc[train_set.CELL_LINE_NAME]
    tx_mean = train_exp.mean(0)
    tx_std = train_exp.std(0)
    
    ## IC50
    ic50_mean = train_set.groupby("DRUG_NAME")["LN_IC50"].mean().to_dict()
    ic50_std = train_set.groupby("DRUG_NAME")["LN_IC50"].std().to_dict()
    
    # Read files
    test_set = pd.read_csv(f"{fold}/test_set.csv", index_col=0)
    gex = expression_data.loc[test_set.CELL_LINE_NAME]
    gex = (gex-tx_mean)/tx_std
    
    smiles = vector_smiles.loc[test_set.DRUG_NAME]
    
    test_set["Y_TRUE"] = (test_set["LN_IC50"]-test_set["DRUG_NAME"].map(ic50_mean))/test_set["DRUG_NAME"].map(ic50_std)
    
    test_ds = tf.data.Dataset.from_tensor_slices(((gex.to_numpy(), smiles.to_numpy()), )).batch(batch_size)
    
    # Load model
    model = load_model(f"models_{cancer_type}_NBS_cells/screendl_model_{n}.keras")
    
    # Prediction
    y_hat = model.predict(test_ds)
    
    y_hat = y_hat.astype(np.float64)
    
    test_set["Y_HAT"] = y_hat

    pathway = (test_set[["DRUG_NAME","PATHWAY_NAME"]]
             .drop_duplicates())
    
    tcga_desc = (test_set[["CELL_LINE_NAME","TCGA_DESC"]]
             .drop_duplicates())
    
    # Computing metrics
    per_path_pcc = (test_set
                    .groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    per_path_pcc = (per_path_pcc
                    .merge(pathway, on="DRUG_NAME", how="left")
                    .dropna(subset=["PATHWAY_NAME"])
                    .groupby("PATHWAY_NAME")["PCC"]
                    .median()
                    )

    per_path_pcc_cv[fold_name] = per_path_pcc

    per_tcga_pcc = (test_set
                    .groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    
    per_tcga_pcc = (per_tcga_pcc.
                    merge(tcga_desc, on="CELL_LINE_NAME", how="left").
                    dropna(subset="TCGA_DESC")
                    .groupby("TCGA_DESC")["PCC"]
                    .median()
                    )

    per_tcga_pcc_cv[fold_name] = per_tcga_pcc

2025-10-21 21:22:36.696616: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-10-21 21:22:36.952486: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudnn.so.8'; dlerror: libcudnn.so.8: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda-11.7/lib64
2025-10-21 21:22:36.952522: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1934] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2025-10-21 21:22:36.963946: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorF

135/135 [==============================] - 2s 7ms/step


/tmp/ipykernel_56589/2681495236.py:54: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_56589/2681495236.py:69: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))


136/136 [==============================] - 2s 7ms/step


/tmp/ipykernel_56589/2681495236.py:54: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_56589/2681495236.py:69: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))


...

/tmp/ipykernel_56589/2681495236.py:54: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_56589/2681495236.py:69: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))


135/135 [==============================] - 1s 7ms/step


/tmp/ipykernel_56589/2681495236.py:54: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_56589/2681495236.py:69: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))


134/134 [==============================] - 2s 7ms/step


/tmp/ipykernel_56589/2681495236.py:54: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_56589/2681495236.py:69: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))


In [6]:
exclude = ["Unclassified", "Other", "Other, kinases", "Chromatin other"]

In [28]:
path = (pd.DataFrame(per_path_pcc_cv)
 .groupby(level=0)
 .median()
 .drop(exclude)
#  .median(axis=1)
 .T
 .reset_index(drop=True)
 )

path["fold"] = path.index

path = (path
        .melt(id_vars="fold", var_name="PATHWAY_NAME", value_name="MCorrelation")
        )

path = path.assign(model="ScreenDL")
path.to_csv("predictions_pancancer_NBS_cells/msigdb_fixed-drug_PATHWAY_CV.csv")

In [31]:
tcga = (pd.DataFrame(per_tcga_pcc_cv)
        .dropna()
        # .median(axis=1)
        .drop(["UNCLASSIFIED"])
        .rename_axis("")
 )

tcga = pd.DataFrame(tcga).T
tcga = tcga.reset_index(drop=True)

tcga["fold"] = tcga.index

tcga = (tcga
        .melt(id_vars = "fold", var_name="TCGA_DESC", value_name="MCorrelation")
        )

tcga = tcga.assign(model="ScreenDL")
tcga.to_csv("predictions_pancancer_NBS_cells/msigdb_fixed-cell_TCGA_CV.csv")

tcga

,fold,TCGA_DESC,MCorrelation,model
0,0,ALL,0.504906,ScreenDL
1,1,ALL,0.237093,ScreenDL
2,2,ALL,0.285342,ScreenDL
3,3,ALL,0.255369,ScreenDL
4,4,ALL,0.459184,ScreenDL
...,...,...,...,...
115,5,SKCM,0.517630,ScreenDL
116,6,SKCM,0.505041,ScreenDL
117,7,SKCM,0.398055,ScreenDL
118,8,SKCM,0.421837,ScreenDL


# Metrics comparison 

# Organoid dataset

In [21]:
gdsc_gex = pd.read_csv("../../output/regression/msigdb_GEX_data_filtered_logCPM.csv", header=0, index_col=0)

In [22]:
gdsc_gex.head()

,A2M,AAAS,AADAT,AARS1,ABAT,ABCA1,ABCA2,ABCA3,ABCA4,ABCA5,...,ZNF292,ZNF365,ZNF639,ZNF707,ZNFX1,ZNRF4,ZPBP,ZW10,ZWINT,ZYX
model_name,,,,,,,,,,,,,,,,,,,,,
22RV1,3.399924,6.520648,3.603538,8.643074,2.971665,0.962100,7.976382,7.476147,0.062735,4.646412,...,5.903139,1.381382,5.294138,3.299304,5.120702,0.000000,0.0,5.302827,7.632962,4.821401
23132-87,0.025782,5.474294,0.076004,9.250811,1.764760,0.994019,7.163113,3.679261,0.076004,4.127690,...,5.677038,0.025782,5.085057,4.354442,6.881830,0.000000,0.0,5.407221,6.560479,5.913348
42-MG-BA,0.211171,5.790924,2.908818,8.096846,4.242014,2.461569,4.276452,0.324470,0.876116,1.898000,...,5.674195,2.380913,4.345195,3.933675,7.137505,0.022564,0.0,5.603927,6.532902,7.476502
451Lu,9.054014,5.793989,3.324716,7.969485,0.281974,5.013172,6.134654,0.977054,0.081803,4.314850,...,4.531588,3.101720,5.226473,3.926546,6.222958,0.000000,0.0,4.471016,6.060170,7.353454
5637,0.197640,5.671271,4.719549,6.429777,0.702504,5.548689,3.983275,0.180751,0.163662,3.686967,...,4.990610,1.760936,4.696712,3.873259,6.263531,0.000000,0.0,5.516239,6.461589,7.794627


In [23]:
pdo_gex = pd.read_csv(
    "../../data/validation_data/PDOs/integrated_data/pdo_logCPM.csv",
    index_col=0
)

In [24]:
pdo_gex = pdo_gex.loc[
    :,pdo_gex.columns.intersection(gdsc_gex.columns)]
pdo_gex = pdo_gex.reindex(columns=gdsc_gex.columns, fill_value=0.0)
pdo_gex.shape

(54, 4377)

In [25]:
pdo_dr = pd.read_csv(
    "../../data/validation_data/PDOs/integrated_data/dose_response_pdo.csv"
)
pdo_dr = pdo_dr.loc[pdo_dr.seen_before=="yes",:]

In [26]:
smiles = pd.read_csv(
    "../../data/validation_data/PDOs/integrated_data/vector_smiles.csv",
    index_col=0
)

In [27]:
epochs= 100
batch_size= 256
learning_rate= 0.0001
weight_decay= 1e-4
earl_stopping=True


In [28]:
cancer_type = "pancancer"
root = f"../../output/regression/cross_validation/{cancer_type}/NBS_cells"

per_drug_pcc_cv = {}
per_line_pcc_cv = {}

per_drug_precision_cv = {}
per_cell_precision_cv = {}

per_drug_ndcg_cv = {}
per_cell_ndcg_cv = {}

for n in range(10):
    fold_name = f"fold_{n}"
    fold = f"{root}/fold_{n}"
    
    # Train set
    ## Indices
    train_set = pd.read_csv(f"{fold}/train_set.csv", index_col=0)
    
    ## GEX
    train_exp = gdsc_gex.loc[train_set.CELL_LINE_NAME]
    tx_mean = train_exp.mean(0)
    tx_std = train_exp.std(0)
    
    ## IC50
    ic50_mean = train_set.groupby("DRUG_NAME")["LN_IC50"].mean().to_dict()
    ic50_std = train_set.groupby("DRUG_NAME")["LN_IC50"].std().to_dict()
    
    
    
    ## Y TRUE
    pdo_dr["Y_TRUE"] = (pdo_dr["LogIC50"]-pdo_dr["Drug"].map(ic50_mean))/pdo_dr["Drug"].map(ic50_std).to_numpy()
    
    gex = pdo_gex.loc[pdo_dr.Line,:]
    gex = (gex-tx_mean)/tx_std
    sm = smiles.loc[pdo_dr.Drug, :]
    
    test_ds = tf.data.Dataset.from_tensor_slices(((gex.to_numpy(), sm.to_numpy()),)).batch(batch_size)
    
    # Load model
    model = load_model(f"models_{cancer_type}_NBS_cells/screendl_model_{n}.keras")
    
    y_hat = model.predict(test_ds)
    y_hat = y_hat.astype(np.float64)
    
    pdo_dr["Y_HAT"] = y_hat
    
    per_drug_pcc = (pdo_dr.groupby("Drug")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Drug")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())

    per_drug_pcc_cv[fold_name] = [per_drug_pcc["PCC"].median()]

    per_line_pcc = (pdo_dr.groupby("Line")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Line")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index()
                    .sort_values("PCC", ascending=True)
                  )

    per_line_pcc_cv[fold_name] = [per_line_pcc["PCC"].median()]

    ##############################
    # PRECISION
    ##############################
    per_drug_pcc = (test_set.groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_drug_precision_cv[fold_name] = [per_drug_pcc["precision@q25"].median()]

    per_cell_pcc = (test_set.groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_cell_precision_cv[fold_name] = [per_cell_pcc["precision@q25"].median()]

    ##############################
    # NDCG
    ##############################
    per_drug_pcc = (test_set.groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_drug_ndcg_cv[fold_name] = [per_drug_pcc["ndcg@q25"].median()]

    per_cell_pcc = (test_set.groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_cell_ndcg_cv[fold_name] = [per_cell_pcc["ndcg@q25"].median()]
    
    print(f"Finished fold: {n}")

9/9 [==============================] - 0s 7ms/step


/tmp/ipykernel_28463/3372502760.py:52: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/3372502760.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/3372502760.py:73: Deprecation

Finished fold: 0
9/9 [==============================] - 0s 7ms/step


/tmp/ipykernel_28463/3372502760.py:52: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/3372502760.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/3372502760.py:73: Deprecation

Finished fold: 1
9/9 [==============================] - 0s 6ms/step


...

/tmp/ipykernel_28463/3372502760.py:52: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/3372502760.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/3372502760.py:73: Deprecation

Finished fold: 8
9/9 [==============================] - 0s 6ms/step


/tmp/ipykernel_28463/3372502760.py:52: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/3372502760.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_28463/3372502760.py:73: Deprecation

Finished fold: 9


/tmp/ipykernel_28463/3372502760.py:103: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))


In [29]:
per_drug_pcc_cv = pd.DataFrame(per_drug_pcc_cv)
per_drug_pcc_cv.index = ["ScreenDL"]
per_drug_pcc_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
ScreenDL,-0.00871,0.051965,0.070208,-0.071838,0.130668,0.14071,0.239581,0.109375,-0.069385,0.039733


In [30]:
M = np.median(per_drug_pcc_cv)
sem = stats.sem(per_drug_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.061086145445936844, low=-0.008643702242458175, high=0.13081599313433187


In [12]:
per_line_pcc_cv = pd.DataFrame(per_line_pcc_cv)
per_line_pcc_cv.index = ["ScreenDL"]
per_line_pcc_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
ScreenDL,0.0786,0.249275,0.113607,0.027864,0.103968,0.055806,0.109403,0.022201,0.02658,0.185182


In [13]:
M = np.median(per_line_pcc_cv)
sem = stats.sem(per_line_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.09128405880233295, low=0.03860260247933919, high=0.1439655151253267


In [18]:
per_drug_pcc_cv.to_csv(
    "predictions_pancancer_NBS_cells/pdo_fixed-drug_evaluation.csv"
)

In [19]:
per_line_pcc_cv.to_csv(
    "predictions_pancancer_NBS_cells/pdo_fixed-line_evaluation.csv"

)

In [31]:
per_drug_precision_cv = pd.DataFrame(per_drug_precision_cv)
per_drug_precision_cv.index = ["ScreenDL"]
per_drug_precision_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
ScreenDL,0.454545,0.454545,0.454545,0.454545,0.454545,0.454545,0.454545,0.454545,0.454545,0.454545


In [32]:
per_drug_precision_cv.to_csv(
    "predictions_pancancer_NBS_cells/precision_pdo_fixed-drug_CV.csv"
)

In [33]:
per_drug_ndcg_cv = pd.DataFrame(per_drug_ndcg_cv)
per_drug_ndcg_cv.index = ["ScreenDL"]
per_drug_ndcg_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
ScreenDL,0.449285,0.449285,0.449285,0.449285,0.449285,0.449285,0.449285,0.449285,0.449285,0.449285


In [34]:
per_drug_ndcg_cv.to_csv(
    "predictions_pancancer_NBS_cells/ndcg_pdo_fixed-drug_CV.csv"
)

In [35]:
per_cell_precision_cv = pd.DataFrame(per_cell_precision_cv)
per_cell_precision_cv.index = ["ScreenDL"]
per_cell_precision_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
ScreenDL,0.411111,0.411111,0.411111,0.411111,0.411111,0.411111,0.411111,0.411111,0.411111,0.411111


In [39]:
per_cell_precision_cv.to_csv(
    "predictions_pancancer_NBS_cells/precision_pdo_fixed-line_CV.csv"
)

In [37]:
per_cell_ndcg_cv = pd.DataFrame(per_cell_ndcg_cv)
per_cell_ndcg_cv.index = ["ScreenDL"]
per_cell_ndcg_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
ScreenDL,0.321888,0.321888,0.321888,0.321888,0.321888,0.321888,0.321888,0.321888,0.321888,0.321888


In [40]:
per_cell_ndcg_cv.to_csv(
    "predictions_pancancer_NBS_cells/ndcg_pdo_fixed-line_CV.csv"
)

In [16]:
cancer_type = "pancancer"
root = f"../../output/regression/cross_validation/{cancer_type}/NBS_cells"

per_cancer_pcc_cv = {}
per_lab_pcc_cv = {}

for n in range(10):
    fold_name = f"fold_{n}"
    fold = f"{root}/fold_{n}"
    
    # Train set
    ## Indices
    train_set = pd.read_csv(f"{fold}/train_set.csv", index_col=0)
    
    ## GEX
    train_exp = gdsc_gex.loc[train_set.CELL_LINE_NAME]
    tx_mean = train_exp.mean(0)
    tx_std = train_exp.std(0)
    
    ## IC50
    ic50_mean = train_set.groupby("DRUG_NAME")["LN_IC50"].mean().to_dict()
    ic50_std = train_set.groupby("DRUG_NAME")["LN_IC50"].std().to_dict()
    
    
    
    ## Y TRUE
    pdo_dr["Y_TRUE"] = (pdo_dr["LogIC50"]-pdo_dr["Drug"].map(ic50_mean))/pdo_dr["Drug"].map(ic50_std).to_numpy()
    
    gex = pdo_gex.loc[pdo_dr.Line,:]
    gex = (gex-tx_mean)/tx_std
    sm = smiles.loc[pdo_dr.Drug, :]
    
    test_ds = tf.data.Dataset.from_tensor_slices(((gex.to_numpy(), sm.to_numpy()),)).batch(batch_size)
    
    # Load model
    model = load_model(f"models_{type}_NBS_cells/screendl_model_{n}.keras")
    
    y_hat = model.predict(test_ds)
    y_hat = y_hat.astype(np.float64)
    
    pdo_dr["Y_HAT"] = y_hat
    
    cancer_type = (
        pdo_dr[["Line","TCGA_DESC"]]
        .drop_duplicates()
    )

    labs = (
        pdo_dr[["Line", "Lab"]]
        .drop_duplicates()
    )


    # Computing metrics
    per_cancer_pcc = (pdo_dr.groupby("TCGA_DESC")
                    .filter(lambda x: len(x) >=2)
                    .groupby("TCGA_DESC")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    per_cancer_pcc = (per_cancer_pcc
                      .merge(cancer_type, on="TCGA_DESC", how="left")
                      .dropna(subset=["TCGA_DESC"])
                      .groupby("TCGA_DESC")["PCC"]
                      .median()
                     )

    per_cancer_pcc_cv[fold_name] = per_cancer_pcc

    per_lab_pcc = (pdo_dr.groupby("Lab")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Lab")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    per_lab_pcc = (per_lab_pcc
                   .merge(labs, on="Lab", how="left")
                   .dropna(subset=["Lab"])
                   .groupby("Lab")["PCC"]
                   .median()
                   )

    per_lab_pcc_cv[fold_name] = per_lab_pcc
    
    print(f"Finished fold: {n}")

9/9 [==============================] - 0s 6ms/step
Finished fold: 0


/tmp/ipykernel_25414/358425151.py:58: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_25414/358425151.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))


9/9 [==============================] - 0s 6ms/step
Finished fold: 1


/tmp/ipykernel_25414/358425151.py:58: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_25414/358425151.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))


9/9 [==============================] - 0s 6ms/step
Finished fold: 2


...

/tmp/ipykernel_25414/358425151.py:58: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_25414/358425151.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))


9/9 [==============================] - 0s 6ms/step
Finished fold: 8


/tmp/ipykernel_25414/358425151.py:58: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_25414/358425151.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))


9/9 [==============================] - 0s 6ms/step
Finished fold: 9


/tmp/ipykernel_25414/358425151.py:58: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_25414/358425151.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))


In [17]:
cancer_type = (pd.DataFrame(per_cancer_pcc_cv)
               .groupby(level=0)
               .median()
               .median(axis=1)
               .rename_axis("")
               .T
)

cancer_type.name = "ScreenDL"

cancer_type


BLCA    0.209691
COAD    0.020824
HNSC    0.194371
PDAC    0.095119
Name: ScreenDL, dtype: float64

In [20]:
cancer_type.to_csv(
    "predictions_pancancer_NBS_cells/pdo_fixed-cancer_CV.csv"
)